# FemScan AI ? Colab Quick Test
This notebook lets you run the Streamlit app in Colab and automatically download the model weights from Google Drive.

**Prereqs**
- Share the `.pt` file with **Anyone with the link can view**.
- Have the repository available in Colab (clone or upload).


## 1) Set Repo Location
Edit `REPO_URL` if you want to clone from GitHub. If you uploaded the repo manually, set `REPO_DIR` to that folder.


In [ ]:
# @title Repo Setup
REPO_URL = ""  # e.g. https://github.com/your-org/femscan-ai
REPO_DIR = '/content/femscan-ai'

import os, subprocess

if REPO_URL and not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

if not os.path.exists(REPO_DIR):
    raise FileNotFoundError(f"Repo not found at {REPO_DIR}. Upload or clone it first.")

print('Repo ready:', REPO_DIR)


## 2) Install Dependencies


In [ ]:
# @title Install Requirements
import subprocess
subprocess.run(['pip', '-q', 'install', '-r', f'{REPO_DIR}/requirements.txt'], check=True)
print('Dependencies installed.')


## 3) Set Model Weights (Google Drive)
Add your Drive file ID below.


In [ ]:
# @title Set Drive File ID
import os
os.environ['FEMSCAN_DRIVE_FILE_ID'] = '1njtElasnkalLgpccfv1seBT4JmqOQYRR'  # replace if needed
print('FEMSCAN_DRIVE_FILE_ID set')


## 4) (Optional) Pre-download Weights
The app can download at runtime, but you can prefetch to save time.


In [ ]:
# @title Download Weights Now (Optional)
import os
from pathlib import Path
import requests

def get_confirm_token(response):
    for k, v in response.cookies.items():
        if k.startswith('download_warning'):
            return v
    text = response.text
    marker = 'confirm='
    if marker in text:
        idx = text.find(marker) + len(marker)
        token = ''
        while idx < len(text) and text[idx].isalnum():
            token += text[idx]
            idx += 1
        return token or None
    return None

def save_stream(response, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    with dest.open('wb') as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

def download_from_drive(file_id: str, dest: Path):
    url = 'https://drive.google.com/uc?export=download'
    session = requests.Session()
    response = session.get(url, params={'id': file_id}, stream=True, timeout=120)
    token = get_confirm_token(response)
    if token:
        response = session.get(url, params={'id': file_id, 'confirm': token}, stream=True, timeout=120)
    response.raise_for_status()
    save_stream(response, dest)

file_id = os.environ.get('FEMSCAN_DRIVE_FILE_ID')
if not file_id:
    raise ValueError('FEMSCAN_DRIVE_FILE_ID not set')

dest = Path(REPO_DIR) / 'trained_models' / 'cervical_best.pt'
if dest.exists():
    print('Checkpoint already exists:', dest)
else:
    download_from_drive(file_id, dest)
    print('Downloaded:', dest)


## 5) Launch Streamlit with a Public URL
By default we use **Cloudflare Tunnel** (no auth token required).
If you prefer **ngrok**, see the optional cell below.


In [ ]:
# @title Launch Streamlit with Cloudflare Tunnel (Recommended)
import os, subprocess, time, re
import requests

os.environ['STREAMLIT_SERVER_HEADLESS'] = 'true'
os.environ['STREAMLIT_SERVER_PORT'] = '8501'

# Install cloudflared (direct binary download)
bin_path = '/usr/local/bin/cloudflared'
if not os.path.exists(bin_path):
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(bin_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    os.chmod(bin_path, 0o755)

# Start Streamlit in background
proc = subprocess.Popen(['streamlit', 'run', f'{REPO_DIR}/app.py', '--server.port', '8501', '--server.headless', 'true'])
time.sleep(3)

# Start tunnel
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
public_url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    if not line:
        continue
    s = line.decode('utf-8', errors='ignore')
    match = re.search(r'https://[-\w]+\.trycloudflare\.com', s)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print('Streamlit URL:', public_url)
else:
    print('Tunnel started, but URL not found. Check logs above.')

print('Keep this cell running to keep the app alive.')


## 6) (Optional) ngrok Tunnel
ngrok now requires an authtoken. Use this only if you have a verified ngrok account.


In [ ]:
# @title Launch Streamlit with ngrok (Optional)
# Only use this if you have an ngrok authtoken
import os, subprocess, time
from pyngrok import ngrok

# Install pyngrok if needed
subprocess.run(['pip', '-q', 'install', 'pyngrok'], check=True)

# Set your auth token
ngrok.set_auth_token('YOUR_NGROK_TOKEN')

os.environ['STREAMLIT_SERVER_HEADLESS'] = 'true'
os.environ['STREAMLIT_SERVER_PORT'] = '8501'

proc = subprocess.Popen(['streamlit', 'run', f'{REPO_DIR}/app.py', '--server.port', '8501', '--server.headless', 'true'])
time.sleep(3)

public_url = ngrok.connect(8501)
print('Streamlit URL:', public_url)
print('Keep this cell running to keep the app alive.')
